# CLIFFGUARD — both annotation axes, on a free T4

**This is the only notebook you need to run.** `colab_run.ipynb` and
`colab_round2.ipynb` are earlier rounds whose results are already in the paper;
nothing here depends on re-running them.

**Runtime: `Runtime → Change runtime type → T4 GPU`, then `Run all`.**
A free T4 is enough — that is a design constraint, not an aspiration, and §2
prints the time budget before anything expensive starts.

## What it measures

The published results measure a *change in the model's own decision*, which
cannot say whether the change was good: a new refusal is either the system
working or the system becoming useless, and an unlabelled corpus reports both as
one number. This separates them, with two independent annotations.

| axis | labels from | question |
|---|---|---|
| **prompt**: harmful / benign | published suites, external to this project | should the model have helped? |
| **completion**: refusal / compliance / deflection / disclaimer / unclear | a 7B judge, five-way, first-token argmax | what did it actually do? |

|            | refusal | compliance | deflection | disclaimer | degenerate |
|---|---|---|---|---|---|
| **harmful** | withheld *(desired)* | **safety failure** | partial withhold | non-answer | capability failure |
| **benign**  | **over-refusal** | utility *(desired)* | soft over-refusal | capability failure | capability failure |

The two bold cells point in opposite directions and are **never summed**.

## It survives its own runtime ending

Every stage is a step in a journal on Drive. A step that finished is skipped; one
whose command or output changed is re-run; one whose output vanished is re-run;
inside a step the ladder and grader cache **per scheme**. Near the end of a
session the pipeline declines to *start* a long step, so the runtime is reclaimed
between steps rather than inside one.

**If it disconnects: reconnect and re-run the pipeline cell. That is the whole
recovery procedure.**

## 0 — Environment

Installs only what Colab lacks. `numpy` is deliberately **not** pinned — forcing
`numpy<2` breaks Colab's preinstalled torch through an ABI mismatch.

`CLIFFGUARD_ARTIFACTS` points run directories at Drive. Written relative to the
clone they would live in `/content` and vanish with the session, after the GPU
time that produced them has been spent.

In [ ]:
import os, sys, json, pathlib, subprocess, platform

IN_COLAB   = "google.colab" in sys.modules
REPO_URL   = "https://github.com/parnish007/CLIFFGUARD.git"
BRANCH     = "main"
REPO_DIR   = pathlib.Path("/content/CLIFFGUARD") if IN_COLAB else pathlib.Path.cwd()
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/cliffguard")

if IN_COLAB:
    try:
        from google.colab import drive as _drive
        _drive.mount("/content/drive")
    except Exception as exc:
        # Do NOT quietly continue on ephemeral storage. Everything that makes
        # this notebook resumable -- the journal, the per-scheme caches, the run
        # directories -- lives on Drive. Without it a reconnect starts from
        # zero, and the failure would only become visible hours later.
        raise SystemExit(" ".join([
            f"Google Drive did not mount ({exc}).",
            "This notebook keeps its journal, caches and results on Drive;",
            "without it a disconnect loses everything and the run is not",
            "resumable. Re-run this cell and approve the authorisation prompt.",
            "Mounting Drive is the ONE click this notebook needs.",
        ]))
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1",
                        "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard",
                        f"origin/{BRANCH}"], check=True)
    # transformers is pinned below 5. The 5.x line installs cleanly against
    # older torch and then SEGFAULTS on every from_pretrained -- measured on
    # torch 2.5.1, exit 139, with no model code involved. Colab's torch moves
    # without warning, so this is pinned rather than trusted.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.44,<5", "accelerate", "bitsandbytes",
                    "datasets", "sentencepiece", "protobuf"], check=True)
    # WHERE the model weights go, decided by measured free space rather than by
    # preference. Keeping them on Drive means a reconnect re-uses them instead of
    # re-downloading, which is the single biggest saving on a resumed session.
    # But the default model set plus the 7B judge is about 32 GB, and a free
    # Google Drive is 15 GB in total -- so unconditionally caching to Drive fills
    # it and the run dies part-way with a disk error, having also filled the
    # user's actual Drive. /content is ephemeral but roughly 78 GB.
    import shutil as _shutil

    _need_gb = 34.0                       # the default MODELS + judge, with slack
    try:
        _free_gb = _shutil.disk_usage(DRIVE_ROOT).free / 1e9
    except OSError:
        _free_gb = 0.0
    if _free_gb >= _need_gb:
        os.environ["HF_HOME"] = str(DRIVE_ROOT / "hf")
        print(f"[hf] weights -> Drive ({_free_gb:.0f} GB free): a reconnect will "
              "re-use them")
    else:
        os.environ["HF_HOME"] = "/content/hf"
        print(f"[hf] weights -> /content: Drive has {_free_gb:.0f} GB free and "
              f"this needs about {_need_gb:.0f} GB.")
        print("     Ephemeral, so a reconnect re-downloads them. That is the "
              "cheaper mistake:\n     filling Drive fails the run AND leaves you "
              "with no space.")
    pathlib.Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

# The clone comes from the REMOTE. If the branch has not been pushed since this
# notebook was written, the checkout is missing the modules below and the
# preflight dies three cells later with a bare ModuleNotFoundError that says
# nothing about the cause. Name it here instead.
_needs = ["scripts/colab_pipeline.py",
          "scripts/classify_completion_taxonomy.py",
          "scripts/analyse_matrix.py",
          "scripts/download_eval_suites.py",
          "scripts/analyse_labelled.py"]
_absent = [f for f in _needs if not (REPO_DIR / f).exists()]
if _absent:
    _head = subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"],
                           capture_output=True, text=True).stdout.strip()
    raise SystemExit("\n".join([
        "This notebook needs code that is NOT in the cloned repository.",
        f"  missing    : {_absent}",
        f"  clone is at: {_head}",
        f"  branch     : {BRANCH} of {REPO_URL}",
        "",
        "The clone reflects what has been PUSHED, not what is on a laptop.",
        "Push the branch containing these files and re-run, or point",
        "REPO_URL / BRANCH at one that has them.",
    ]))
print("repo has every module this notebook imports")

ARTIFACTS = DRIVE_ROOT / "artifacts"
os.environ["CLIFFGUARD_ARTIFACTS"] = str(ARTIFACTS)
(ARTIFACTS / "runs").mkdir(parents=True, exist_ok=True)
CACHE_ROOT = DRIVE_ROOT / "cache_labelled"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

import torch
VRAM_GB = (torch.cuda.get_device_properties(0).total_memory / 1e9
           if torch.cuda.is_available() else 0.0)
print(f"repo      : {REPO_DIR}")
print(f"drive     : {DRIVE_ROOT}")
print(f"hf cache  : {os.environ.get('HF_HOME', '(default, not on Drive)')}")
print(f"python    : {platform.python_version()}   torch {torch.__version__}")
print(f"gpu       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}"
      f"   {VRAM_GB:.1f} GB")
if not torch.cuda.is_available():
    print("!! no GPU: Runtime -> Change runtime type -> T4 GPU")

## 1 — Configuration, and the cost of it

Set before the preflight, because the preflight checks these values.

**`MODELS`** — three families by default. Qwen and Phi are the paper's; SmolLM2
is added because two families cannot distinguish "a property of quantization"
from "a property of these two checkpoints", and it is Apache-2.0 and **ungated**,
so `Run all` needs no HuggingFace token. Llama-3.2 and Gemma-2 are listed but
commented out: both are gated, and a notebook that stops to ask for a token is
not a notebook that survives a reconnect.

**`CORPUS`** — **XSTest**, whose 200 harmful and 250 benign prompts were built
*together* as matched contrasts. The 2×2 is paired against each run's own FP16
baseline, so both classes must be in the **same** run, and XSTest is the only
suite that ships both. The `paired-*` corpora join a harmful suite to a benign
one to get more prompts — but that makes prompt class perfectly confounded with
authorship, so a class-level difference is also a difference between two research
groups. The within-prompt transitions are unaffected; the per-class baseline
rates are not. Prefer XSTest unless you need the volume.

**`N_PER_CLASS`** is per harmfulness class, and the loader interleaves before
truncating — the files are class-ordered, so truncating first would silently give
a run with zero harmful prompts and a table that still looks fine.

In [ ]:
MODELS = [
    ("qwen3b",  "Qwen/Qwen2.5-3B-Instruct"),          # Qwen        ~6.2 GB fp16
    ("phi35",   "microsoft/Phi-3.5-mini-instruct"),   # Microsoft   ~7.6 GB fp16
    ("smol17",  "HuggingFaceTB/SmolLM2-1.7B-Instruct"),  # HFTB     ~3.4 GB fp16
    # Gated -- need a HF token, so they are opt-in rather than default:
    # ("llama32", "meta-llama/Llama-3.2-3B-Instruct"),
    # ("gemma2",  "google/gemma-2-2b-it"),
]

# XSTest: 200 harmful and 250 benign, built TOGETHER as matched contrasts by one
# group. That matters more than corpus size here. Every "paired-*" corpus takes
# its harmful prompts from one suite and its benign prompts from another, which
# makes prompt class perfectly confounded with authorship -- any difference
# between the classes is also a difference between two construction procedures.
# The paired transitions are within-prompt so the confound cancels there, but the
# full-precision baseline rates printed per class are a cross-class comparison
# and it does not. Use a pairing only when more prompts of one class are needed
# than XSTest has:
#   paired-harmbench-orbench    (200 harmful  + 1319 benign)
#   paired-advbench-orbench     (520 harmful  + 1319 benign)
#   paired-strongreject-xstest  (313 harmful  +  250 benign)
CORPUS = "xstest"

JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
JUDGE_4BIT  = VRAM_GB < 20          # a T4 cannot hold 7B in fp16
N_PER_CLASS = 150                   # -> 300 prompts per run
BITS        = [8, 7, 6, 5, 4, 3, 2] # the paper's seven rungs
MAX_NEW     = 48
BATCH       = 16 if VRAM_GB >= 20 else 8
JUDGE_BATCH = 8  if VRAM_GB >= 20 else 4

# Stop BETWEEN steps rather than be killed inside one. Free Colab reclaims at
# roughly 4 h, Pro at 12; set slightly under whichever you have.
DEADLINE_HOURS = 3.5 if VRAM_GB < 20 else 11.0

RUNS_ROOT = ARTIFACTS / "runs"
JOURNAL   = DRIVE_ROOT / "journal_labelled.json"

# ---- what this will cost ------------------------------------------------
# Calibrated from this project's own measured throughput on a T4: roughly
# 0.8 s per prompt per scheme for a 3B model at 48 new tokens, and about
# 0.30 s per (prompt, scheme) pair for the 7B judge in NF4. Rough by design --
# it exists to catch "this cannot finish" before the download, not to be exact.
n_prompts = N_PER_CLASS * 2
n_schemes = len(BITS) + 1
# Generation dominates, but it is not the only cost, and an estimate that omits
# the rest is the kind that says "fits" right up until it does not:
#   * every scheme loads a model and RTN-quantizes it   ~1.5 min each
#   * every completion is scored for NLL under FP16     ~0.05 s each
gen_min   = n_prompts * n_schemes * 0.8 / 60
load_min  = n_schemes * 1.5
nll_min   = n_prompts * n_schemes * 0.05 / 60
judge_min = (n_prompts * n_schemes * 0.30 / 60) + 3     # +3 for the 7B load
per_model = gen_min + load_min + nll_min + judge_min
total_h   = per_model * len(MODELS) / 60

print(f"models     : {[m for m, _ in MODELS]}")
print(f"corpus     : {CORPUS}   ({n_prompts} prompts = {N_PER_CLASS} per class)")
print(f"rungs      : {BITS}  -> {n_schemes} schemes incl. FP16")
print(f"judge      : {JUDGE_MODEL}  (4-bit: {JUDGE_4BIT})")
print()
print(f"estimated  : ~{gen_min:.0f} min generate + ~{load_min:.0f} min loads + "
      f"~{nll_min:.0f} min NLL + ~{judge_min:.0f} min judge")
print(f"             = ~{per_model:.0f} min per model  (rough: it exists to catch "
      f"'cannot finish' before the download)")
print(f"             ~{total_h:.1f} h for {len(MODELS)} models")
print(f"session cap: {DEADLINE_HOURS} h -> "
      + ("fits in one session" if total_h <= DEADLINE_HOURS else
         f"needs ~{-(-total_h // DEADLINE_HOURS):.0f} sessions; "
         "re-run the pipeline cell after each reconnect"))
print()
print("three optimisations are why this fits a free tier at all:")
print("  --no-activations    the residual stream is a second full forward over")
print("                      every prompt and ONLY the probe arm reads it")
print("  one judge pass      the 5-way grader also emits the 3-way labels, since")
print("                      3-way REFUSE == REFUSE+DEFLECT+DISCLAIM exactly")
print("  a paired corpus     one ladder fills the whole 2x2 instead of two")

## 2 — PREFLIGHT

Seconds, on CPU, before any download or GPU time. It imports every symbol the
run uses, exercises the resumable runner end-to-end on a toy step, checks the
matrix cells are counted in the right corners, and — the expensive one to get
wrong — verifies the five taxonomy labels have **distinct first tokens** under
the judge's tokenizer. First-token argmax over labels sharing a first piece is
not a five-way choice at all; it produces verdicts that look fine and mean
nothing.

`PREFLIGHT OK` means nothing below can die on an import, a signature, or a
tokenizer surprise. If it fails, **stop**.

In [ ]:
import numpy as np, tempfile, pathlib as _pl

failures = []
def check(label, fn):
    try:
        fn(); print(f"  ok    {label}")
    except Exception as exc:
        failures.append(f"{label}: {type(exc).__name__}: {exc}")
        print(f"  FAIL  {label}: {type(exc).__name__}: {exc}")

print("imports")
from scripts.colab_pipeline import Pipeline, Step, latest_run
from scripts.classify_completion_taxonomy import (
    LABELS, CLASS_OF, label_first_token_ids, resolve, gate_mask)
from scripts.analyse_matrix import (CLASSES, COMPLIED, NOT_COMPLIANCE,
                                    contingency, paired, transition_matrix)
from scripts.analyse_labelled import HARMFUL, BENIGN
from scripts.download_eval_suites import PAIRS
from scripts.run_behavioural_ladder import is_degenerate
from scripts.run_local_ladder import load_labelled_prompts
from cliffguard.eval.storage import ARTIFACTS_ROOT
print("  ok    every symbol the run uses")

check("run directories point at Drive",
      lambda: None if str(DRIVE_ROOT) in str(ARTIFACTS_ROOT)
      else (_ for _ in ()).throw(AssertionError(f"{ARTIFACTS_ROOT} not under {DRIVE_ROOT}")))

def _pipeline_roundtrip():
    with tempfile.TemporaryDirectory() as d:
        d = _pl.Path(d); marker = d / "ran.txt"
        for _ in range(2):
            p = Pipeline(journal=d / "j.json", log_dir=d / "logs", cwd=d)
            p.add(Step("t", [sys.executable, "-c", f"open(r'{marker}','a').write('x')"]))
            assert p.run() == 0
        assert marker.read_text() == "x", "a finished step ran twice"
check("resumable runner skips finished work", _pipeline_roundtrip)

def _taxonomy_tokens():
    from transformers import AutoTokenizer
    ids = label_first_token_ids(AutoTokenizer.from_pretrained(JUDGE_MODEL))
    assert len(set(ids)) == len(LABELS)
    print(f"        {dict(zip(LABELS, ids))}")
check("five taxonomy labels have distinct first tokens", _taxonomy_tokens)

def _gate_then_judge():
    texts = ["I cannot help with that.", "@@@@ @@@@ @@@@ @@@@ @@@@ @@@@", ""]
    ok = gate_mask(texts, np.array([1.0, 1.0, 1.0]), 5.0)
    assert list(ok) == [True, False, False], f"gate misfired: {ok}"
    assert list(resolve(["REFUSE", "COMPLY", "COMPLY"], ok)) == \
        ["refusal", "degenerate", "degenerate"]
check("degeneracy is decided before the judge's verdict", _gate_then_judge)

def _matrix_cells():
    harm = np.array([HARMFUL] * 4 + [BENIGN] * 4)
    labels = {"FP16":   np.array(["refusal", "refusal", "compliance", "compliance",
                                  "compliance", "compliance", "refusal", "refusal"]),
              "RTN_4B": np.array(["compliance", "refusal", "compliance", "refusal",
                                  "refusal", "compliance", "refusal", "compliance"])}
    r = paired(labels, harm)[0]
    assert (r["safety_lost"], r["safety_recovered"]) == (1, 1), r
    assert (r["utility_lost"], r["utility_regained"]) == (1, 1), r
    assert r["n_harmful"] == 4 and r["n_benign"] == 4, "denominator is not the class"
check("the four matrix cells are counted in the right corners", _matrix_cells)

def _endpoint_partitions():
    assert set(COMPLIED) | set(NOT_COMPLIANCE) == set(CLASSES)
    assert not set(COMPLIED) & set(NOT_COMPLIANCE)
check("the endpoint partitions every completion class", _endpoint_partitions)

check("the configured corpus is one the downloader builds",
      lambda: None if (CORPUS in PAIRS or CORPUS in
                       {"xstest", "harmbench", "advbench", "strongreject",
                        "or-bench-hard", "or-bench-toxic"})
      else (_ for _ in ()).throw(AssertionError(f"unknown corpus {CORPUS}")))

print()
if failures:
    raise SystemExit("PREFLIGHT FAILED:\n  " + "\n  ".join(failures))
print("PREFLIGHT OK")

## 3 — Suites

Fetched from the canonical sources by the repository's own downloader.
HarmBench, AdvBench and StrongREJECT come from their GitHub CSVs rather than the
HuggingFace mirrors, which are gated. XSTest is split into halves by the suite's
own `type` field.

The downloader then assembles the **paired** corpora: a harmful suite joined to a
benign one, so a single ladder fills the whole matrix. Everything is hashed into
`MANIFEST.json`, so a re-download that changes anything is visible.

In [ ]:
# The pipeline below has a `suites` step that runs this same downloader, and it
# is the one that records the result in the journal. Fetch here only if the file
# is absent, so `Run all` does not pay for it twice.
_manifest_path = REPO_DIR / "data/eval_suites/MANIFEST.json"
if not _manifest_path.exists():
    subprocess.run([sys.executable, "scripts/download_eval_suites.py",
                    "--download", "--suites", CORPUS, "--pairs"],
                   check=True, cwd=REPO_DIR)
else:
    print("suites already present; the pipeline step will verify them")
manifest = json.loads(_manifest_path.read_text())
print(f"\n{'suite':30s} {'n':>6s} {'harmful':>8s} {'benign':>7s}  provenance")
print("-" * 96)
for s in manifest["suites"]:
    print(f"{s['suite']:30s} {s['n']:6d} {s['counts']['harmful']:8d} "
          f"{s['counts']['benign']:7d}  {s['provenance'][:44]}")
if manifest["failed"]:
    print("\nFAILED:", manifest["failed"])

chosen = next((s for s in manifest["suites"] if s["suite"] == CORPUS), None)
assert chosen, f"{CORPUS} was not built -- see the failures above"
assert chosen["counts"]["harmful"] and chosen["counts"]["benign"], \
    f"{CORPUS} is single-class and cannot fill the 2x2"
print(f"\nusing {CORPUS}: {chosen['counts']['harmful']} harmful / "
      f"{chosen['counts']['benign']} benign available; "
      f"taking {N_PER_CLASS} per class")

## 4 — The pipeline

**This is the cell to re-run after a disconnect.** It builds every step, skips
what is done, and resumes what is not.

Two steps per model, not three. The five-way grader also writes the three-way
labels `analyse_labelled.py` reads, collapsed from its own verdicts — the
three-way template defines `REFUSE` as *"declined, deflected, redirected, gave a
safety warning … or said it cannot or will not help"*, which is exactly
`REFUSE + DEFLECT + DISCLAIM`. Running it separately would be the same 7B sweep
twice. The output records that these are collapsed rather than independent, so
nothing can mistake one for the other.

Grading steps take the run directory as an argument, and a run directory is named
with the timestamp at which it was created — so they are built lazily and
resolved by label when they start. That is how a resumed session finds the ladder
output an earlier session produced.

In [ ]:
from scripts.colab_pipeline import Pipeline, Step, latest_run

pipe = Pipeline(journal=JOURNAL, log_dir=DRIVE_ROOT / "logs_labelled",
                cwd=REPO_DIR, deadline_hours=DEADLINE_HOURS)

# Only the corpus this run uses. The downloader fetches all six by default and
# returns non-zero if ANY source is unreachable -- so an outage at a suite this
# run never opens would abort the whole notebook.
pipe.add(Step("suites",
              [sys.executable, "scripts/download_eval_suites.py",
               "--download", "--suites", CORPUS, "--pairs"],
              produces=REPO_DIR / "data/eval_suites/MANIFEST.json",
              estimated_minutes=3, timeout_s=1800))

for model_key, model_id in MODELS:
    label = f"lab-{model_key}-{CORPUS}"
    pipe.add(Step(
        f"ladder-{label}",
        [sys.executable, "scripts/run_behavioural_ladder.py",
         "--model", model_id,
         "--prompts", f"data/eval_suites/{CORPUS}.jsonl",
         "--n", str(N_PER_CLASS), "--bits", *map(str, BITS),
         "--max-new-tokens", str(MAX_NEW), "--batch-size", str(BATCH),
         # The residual stream is a second full forward over every prompt and
         # only the probe arm reads it. Nothing downstream here does.
         "--no-activations",
         "--cache", str(CACHE_ROOT / f"{model_key}_{CORPUS}"),
         "--label", label],
        estimated_minutes=int(gen_min), timeout_s=4 * 3600))

    pipe.add(Step(
        f"grade-{label}",
        (lambda lab=label: [
            sys.executable, "scripts/classify_completion_taxonomy.py",
            str(latest_run(RUNS_ROOT, lab)),
            "--judge-model", JUDGE_MODEL, "--batch-size", str(JUDGE_BATCH),
            # Explicit although it is the default: the journal fingerprints the
            # COMMAND, so a step recorded before this flag existed would
            # otherwise be skipped and the three-way caches never written.
            "--emit-three-way",
            *(["--judge-4bit"] if JUDGE_4BIT else [])]),
        produces=(lambda lab=label: latest_run(RUNS_ROOT, lab)
                  / "results" / "completion_taxonomy.json"),
        estimated_minutes=int(judge_min), timeout_s=3 * 3600))

pipe.add(Step("analyse-labelled",
              [sys.executable, "scripts/analyse_labelled.py",
               "--runs", str(RUNS_ROOT), "--include", "*lab-*",
               "--out", str(DRIVE_ROOT / "labelled_stats.json")],
              estimated_minutes=2, timeout_s=1800))

pipe.add(Step("analyse-matrix",
              [sys.executable, "scripts/analyse_matrix.py",
               "--runs", str(RUNS_ROOT), "--include", "*lab-*",
               "--out", str(DRIVE_ROOT / "matrix_stats.json")],
              estimated_minutes=2, timeout_s=1800))

failed = pipe.run()
print(f"\n{failed} step(s) failed" if failed else "\nall steps complete")

## 5 — Results

Reads the analysis files and prints the matrix. Nothing is computed here, so
there is no second implementation to drift out of sync with the published one.

In [ ]:
path = DRIVE_ROOT / "matrix_stats.json"
if not path.exists():
    print("matrix_stats.json not written yet -- re-run the pipeline cell")
else:
    data = json.loads(path.read_text())
    for key, block in data.items():
        print(f"\n{'=' * 78}\n{key}   ({block['model']}, {block['corpus']})")
        print(f"{block['n_harmful']} harmful / {block['n_benign']} benign; "
              f"judge {block['judge_model']}; {block['gate']} gate\n{'=' * 78}")
        print(f"{'scheme':9s} | {'SAFETY REGRESSION':^26s} | {'USEFULNESS LOST':^26s}")
        print(f"{'':9s} | {'lost':>5s} {'rec':>4s} {'rate%':>6s} {'p':>6s} | "
              f"{'lost':>5s} {'reg':>4s} {'rate%':>6s} {'p':>6s}")
        print("-" * 70)
        for r in block["paired"]:
            f = lambda v: "  NA  " if v is None else f"{100*v:6.2f}"
            print(f"{r['scheme']:9s} | {r['safety_lost']:5d} {r['safety_recovered']:4d} "
                  f"{f(r['safety_rate'])} {r['safety_p_holm']:6.3f} | "
                  f"{r['utility_lost']:5d} {r['utility_regained']:4d} "
                  f"{f(r['utility_rate'])} {r['utility_p_holm']:6.3f}")
        print(f"\n{'scheme':9s} {'lost':>6s} {'over-refusal':>13s} "
              f"{'capability':>11s} {'over-ref share':>15s}")
        for r in block["paired"]:
            share = ("NA" if r["over_refusal_share"] is None
                     else f"{r['over_refusal_share']:.3f}")
            print(f"{r['scheme']:9s} {r['utility_lost']:6d} {r['over_refusal']:13d} "
                  f"{r['capability_failure']:11d} {share:>15s}")
        print("\nover-refusal = the model declined (refusal + deflection);")
        print("capability   = the model could not (disclaimer + degenerate).")
        print("Opposite diagnoses of the same visible event. Never summed.")

## 6 — Take it home

Everything already lives on Drive and is durable. This packs the small files —
manifests, results, analysis JSON, per-step logs — into one archive. Activations
are not collected at all on this path, and the per-scheme completion caches stay
on Drive rather than being bundled.

In [ ]:
import zipfile, time

stamp  = time.strftime("%Y%m%d-%H%M%S")
bundle = DRIVE_ROOT / f"cliffguard_labelled_{stamp}.zip"
KEEP   = (".json", ".md", ".log")

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for root in (RUNS_ROOT, DRIVE_ROOT / "logs_labelled"):
        if root.exists():
            for p in root.rglob("*"):
                if p.is_file() and p.suffix in KEEP:
                    z.write(p, p.relative_to(DRIVE_ROOT))
    for name in ("matrix_stats.json", "labelled_stats.json", "journal_labelled.json"):
        if (DRIVE_ROOT / name).exists():
            z.write(DRIVE_ROOT / name, name)

print(f"{bundle}  ({bundle.stat().st_size / 1e6:.1f} MB)")
pipe.report()
if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(bundle))
    except Exception as exc:
        print("download failed; the archive is on Drive:", exc)